# Working with Spark SQL Table Properties

Table properties are arbitrary `key = value` metadata that you can attach to a table. They are handy for recording ownership, data-classification labels, lineage hints, or any custom governance metadata **alongside the table itself** — so the context travels with the data instead of living in a separate system.

This notebook walks through the full lifecycle of a table property on the Oracle AI Data Platform (AIDP) lakehouse using a managed **Delta** table:

1. Create a demo table
2. Inspect the engine-managed default properties
3. Store **structured (JSON) metadata** in a property
4. Read properties back (as a DataFrame, as a dict, and a single value)
5. Overwrite an existing property
6. Remove a property

> The cells use plain Spark SQL via `spark.sql(...)`, so the same patterns work in any Spark notebook. The outputs shown below were captured from an actual AIDP run.

**Prerequisite:** an active Spark session (`spark`) — already provided inside an AIDP notebook.

## 1. Create a demo table

We use a fully-qualified name `<catalog>.<schema>.<table>`. `CREATE TABLE IF NOT EXISTS` makes the cell **idempotent** — safe to re-run without error.

In [1]:
# A fully-qualified table name: <catalog>.<schema>.<table>.
# On AIDP this creates a managed Delta table in the lakehouse.
table = "default.default.test_table_prop"

# IF NOT EXISTS makes this cell safe to re-run (idempotent).
spark.sql(f"CREATE TABLE IF NOT EXISTS {table}")

DataFrame[]

## 2. Inspect the default properties

`SHOW TBLPROPERTIES <table>` lists every property as a `(key, value)` row. A freshly created Delta table already carries a few **engine-managed** defaults (the `delta.*` protocol versions and `option.*` entries) — you don't set these yourself.

In [2]:
# SHOW TBLPROPERTIES returns every property as a (key, value) row.
# Newly created Delta tables already carry engine-managed defaults
# (delta.minReaderVersion, delta.minWriterVersion, ...).
props_df = spark.sql(f"SHOW TBLPROPERTIES {table}")
props_df.show(truncate=False)

+---------------------------+-------+
|key                        |value  |
+---------------------------+-------+
|delta.minReaderVersion     |1      |
|delta.minWriterVersion     |2      |
|option.catalog.id          |default|
|option.serialization.format|1      |
+---------------------------+-------+



## 3. Store structured (JSON) metadata in a property

Property **values are always strings**. To attach structured metadata (nested dicts, lists), serialize a Python object to a JSON string with `json.dumps(...)` and store *that string*. You can parse it back later with `json.loads(...)`.

> **Quoting caution:** here the JSON happens to use double quotes, so it nests cleanly inside the single-quoted SQL literal. If a value could itself contain a single quote, escape it (or use a parameterized approach) to avoid breaking the SQL statement.

In [3]:
# --- save json in table property as string ---
# Table-property VALUES are always strings, so to attach structured metadata
# we serialize a Python dict to a JSON string and store that string.
import json

table_property_dict = {
    "owner": "the_owner",
    "labels": ["test", "extra"],
    "position": {"region": {"dc": "region_dc", "az": "region_az"}}
}

# json.dumps -> compact JSON text that round-trips back to a dict later.
table_property_val = json.dumps(table_property_dict)

# ALTER TABLE ... SET TBLPROPERTIES adds (or replaces) the named property.
spark.sql(f"""
    ALTER TABLE {table}
    SET TBLPROPERTIES (
        'custom-details' = '{table_property_val}'
    )
""")

DataFrame[]

## 4. Read the property back

There are a few ways to read properties, depending on what you need.

### As a DataFrame (filter to one key)

In [4]:
# --- specific property as a DataFrame ---
# Filter the SHOW TBLPROPERTIES result down to just the property we care about.
spark.sql(f"SHOW TBLPROPERTIES {table}").filter("key = 'custom-details'").show(truncate=False)

+--------------+-------------------------------------------------------------------------------------------------------------------+
|key           |value                                                                                                              |
+--------------+-------------------------------------------------------------------------------------------------------------------+
|custom-details|{"owner": "the_owner", "labels": ["test", "extra"], "position": {"region": {"dc": "region_dc", "az": "region_az"}}}|
+--------------+-------------------------------------------------------------------------------------------------------------------+



### As a Python dict (handy for programmatic checks)

Collect all rows into a `{key: value}` dict. Because the value was stored as JSON, `json.loads(...)` turns it back into a native Python object.

In [5]:
# --- As a Python dict (handy for programmatic checks) ---
# Materialize all properties into a plain {key: value} dict.
props = {r["key"]: r["value"] for r in spark.sql(f"SHOW TBLPROPERTIES {table}").collect()}
# The stored value is a JSON string -> json.loads() turns it back into a dict.
print(json.loads(props.get("custom-details")))

{'owner': 'the_owner', 'labels': ['test', 'extra'], 'position': {'region': {'dc': 'region_dc', 'az': 'region_az'}}}


### A single property's raw value

`SHOW TBLPROPERTIES <table> ('<key>')` returns just that one property. Note the value comes back as a **`str`** (the raw stored text), so parse it with `json.loads(...)` if you need the structured form.

In [6]:
# --- a single property's value ---
# SHOW TBLPROPERTIES <table> ('<key>') returns only that one property.
prop_val = spark.sql(f"SHOW TBLPROPERTIES {table} ('custom-details')").collect()[0]["value"]
# The value is a str (not a dict) -- parse with json.loads if you need the object.
print(f"{type(prop_val)}: {prop_val}")

<class 'str'>: {"owner": "the_owner", "labels": ["test", "extra"], "position": {"region": {"dc": "region_dc", "az": "region_az"}}}


## 5. Overwrite an existing property

Running `SET TBLPROPERTIES` again with the **same key** simply overwrites the previous value — there is no separate "update" statement.

In [7]:
# Re-running SET TBLPROPERTIES with the same key OVERWRITES the previous value.
spark.sql(f"""
    ALTER TABLE {table}
    SET TBLPROPERTIES (
        'custom-details' = 'overridden-value'
    )
""")

DataFrame[]

In [8]:
# Confirm the value was replaced (now a plain string, no longer JSON).
spark.sql(f"SHOW TBLPROPERTIES {table}").filter("key = 'custom-details'").show(truncate=False)

+--------------+----------------+
|key           |value           |
+--------------+----------------+
|custom-details|overridden-value|
+--------------+----------------+



## 6. Remove a property

`UNSET TBLPROPERTIES` deletes the property. Adding `IF EXISTS` keeps the cell idempotent — it won't error if the key has already been removed.

In [9]:
# UNSET ... IF EXISTS removes the property; IF EXISTS avoids an error
# if the key is already gone (keeps the cell idempotent).
spark.sql(f"ALTER TABLE {table} UNSET TBLPROPERTIES IF EXISTS ('custom-details')")

DataFrame[]

In [10]:
# Verify the property is gone -- the filtered result is now empty.
spark.sql(f"SHOW TBLPROPERTIES {table}").filter("key = 'custom-details'").show(truncate=False)

+---+-----+
|key|value|
+---+-----+
+---+-----+



## Recap & cleanup

- **Set / update:** `ALTER TABLE <t> SET TBLPROPERTIES ('k' = 'v')` (re-running overwrites).
- **Read:** `SHOW TBLPROPERTIES <t>` (all) or `SHOW TBLPROPERTIES <t> ('k')` (one).
- **Remove:** `ALTER TABLE <t> UNSET TBLPROPERTIES IF EXISTS ('k')`.
- **Values are strings** — serialize structured data with `json.dumps` and read it back with `json.loads`.

To clean up the demo table when you're done:

```python
spark.sql(f"DROP TABLE IF EXISTS {table}")
```